In [ ]:
# Komórka 1: Konfiguracja Środowiska i Instalacja Zależności

import os

# Krok 1: Klonowanie repozytorium z modułami
print(" cloning repository...")
!git clone https://github.com/MattyMroz/ColabUniversalDownloader.git

# Przejście do katalogu repozytorium
os.chdir('ColabUniversalDownloader')
print(f" Zmieniono katalog roboczy na: {os.getcwd()}")

# Krok 2: Instalacja zależności systemowych (megatools)
print("\n⚙️ Instalacja 'megatools'...")
!sudo apt-get update -qq > /dev/null
!sudo apt-get install -y megatools -qq > /dev/null
print("✅ 'megatools' zainstalowane pomyślnie.")

# Krok 3: Instalacja zależności z pliku requirements.txt
print("\n🐍 Instalowanie bibliotek Pythona z requirements.txt...")
!pip install -q -r requirements.txt
print("✅ Biblioteki Pythona zainstalowane pomyślnie.")

print("\n\n🎉 Środowisko jest gotowe do testów!")

In [ ]:
# Komórka 2: Import Modułów i Inicjalizacja Narzędzi

import sys
from pprint import pprint

# Dodajemy ścieżkę do repozytorium, aby Python widział nasze moduły w katalogu 'utils'
sys.path.append('/content/ColabUniversalDownloader')

from utils.pixeldrain import PixelDrainDownloader
from utils.mega import MegaDownloader
from utils.google_drive import GoogleDriveManager

# --- Inicjalizacja klas ---
pixeldrain_handler = PixelDrainDownloader()
mega_handler = MegaDownloader()
gdrive_handler = GoogleDriveManager()

print("✅ Moduły zaimportowane, obiekty gotowe do pracy.")
print(f"Google Drive Manager gotowy: {gdrive_handler.is_ready()}")

# --- Prosta funkcja do wyświetlania postępu ---
def simple_progress_callback(log_message):
    """Wyświetla każdą nową linię logu."""
    print(log_message)

In [ ]:
# Komórka 3: Test Modułu `pixeldrain.py`

print("="*50)
print("🚀 TEST 1: POBIERANIE Z PIXELDRAIN 🚀")
print("="*50)

# UWAGA: Użyj działającego linku do testów
PIXEL_URL = "https://pixeldrain.com/u/e75isJ7y" # Przykładowy link, może wygasnąć
filepath = None

try:
    filepath = pixeldrain_handler.download(PIXEL_URL, simple_progress_callback)

    if filepath and os.path.exists(filepath):
        print(f"\n✅ TEST ZAKOŃCZONY SUKCESEM! Plik znajduje się w: {filepath}")
        print(f"   Rozmiar pliku: {os.path.getsize(filepath) / 1024:.2f} KB")
    else:
        print("\n❌ TEST NIEUDANY. Nie udało się pobrać pliku.")

finally:
    # Sprzątanie po teście
    if filepath and os.path.exists(filepath):
        os.remove(filepath)
    #     print(f"\n🧹 Usunięto plik testowy: {filepath}")

In [ ]:
# Komórka 4: Test Modułu `mega.py`

import os
import re
import shutil

print("="*50)
print("🚀 TEST 2: OPERACJE NA MEGA.NZ 🚀")
print("="*50)

# --- Linki testowe (publiczne, bez logowania) ---
MEGA_FOLDER_URL = "https://mega.nz/folder/SVxnDbbI#V5nwTs9FAXNlGzUmWBucIw"  # Folder z dwoma plikami
MEGA_FILE_URL   = "https://mega.nz/file/CVQXjbCZ#KhWlLh7Ec3z0EjcPqVX1_3ZyGQC05xNMs8gb_VYGdxg"  # Pojedynczy plik

# --- Część A: Listowanie zawartości folderu bez logowania ---
print("\n--- TEST 2A: Listowanie folderu ---")
listing_text = None
try:
    listing_text = mega_handler.list_folder_contents(MEGA_FOLDER_URL)
except Exception as e:
    print(f"❌ BŁĄD podczas listowania: {e}")

if listing_text:
    print("\n✅ Zawartość folderu (surowe):\n")
    print(listing_text)
else:
    print("\n⚠️ Brak danych do wyświetlenia (sprawdź link lub sieć)")

# Ekstrakcja nazw plików z listingu
file_names = []
if listing_text:
    for line in listing_text.splitlines():
        if ":" in line:
            name = line.split(":", 1)[0].strip()
            if name:
                file_names.append(name)

if file_names:
    print("\nWykryte pliki w folderze:")
    for idx, name in enumerate(file_names, 1):
        print(f"  [{idx}] {name}")
else:
    print("\nℹ️ Nie udało się pewnie wykryć nazw (spróbuj od razu pobrać cały folder).")

# --- Część B: Pobieranie pojedynczego pliku ---
print("\n--- TEST 2B: Pobieranie pliku ---")
filepath = mega_handler.download_file(MEGA_FILE_URL, progress=simple_progress_callback)
if filepath and os.path.exists(filepath):
    print(f"\n✅ POBRANO PLIK: {filepath}")
    print(f"   Rozmiar: {os.path.getsize(filepath) / (1024*1024):.2f} MB")
else:
    print("\n❌ Nie udało się pobrać pliku (sprawdź URL).")

# --- Część C: Wybór plików do pobrania z folderu (tryb bez TTY) ---
print("\n--- TEST 2C: Wybór plików i pobieranie folderu (bez TTY) ---")
# Uwaga: Colab nie zapewnia pełnego TTY, więc wybór robimy w Pythonie,
# a pobieramy cały folder do katalogu tymczasowego i przenosimy tylko wybrane pliki.

DEST_DIR_ALL = "./mega_tmp"
SELECTED_DIR = "./mega_selected"
os.makedirs(DEST_DIR_ALL, exist_ok=True)
os.makedirs(SELECTED_DIR, exist_ok=True)

selected_names = []
if file_names:
    raw = input("Wybierz pliki do pobrania (indeksy rozdzielone przecinkami, Enter=wszystkie): ").strip()
    if raw:
        try:
            idxs = [int(x) for x in re.split(r"\s*,\s*", raw) if x]
            selected_names = [file_names[i-1] for i in idxs if 1 <= i <= len(file_names)]
        except Exception:
            print("⚠️ Niepoprawny format – pobiorę wszystkie pliki.")
    if not selected_names:
        selected_names = list(file_names)
else:
    print("Brak listy plików – pobiorę cały folder.")

# Pobierz cały folder (bez interaktywnego wyboru)
results = mega_handler.download_folder(MEGA_FOLDER_URL, dest_dir=DEST_DIR_ALL, choose_files=False, progress=simple_progress_callback)

# Wyznacz faktycznie pobrane pliki
downloaded_paths = []
if results:
    downloaded_paths = [p for p in results if os.path.isfile(p)]
else:
    # Fallback: przeskanuj katalog docelowy
    for root, _, files in os.walk(DEST_DIR_ALL):
        for f in files:
            downloaded_paths.append(os.path.join(root, f))

# Przenieś tylko wybrane pliki
moved = []
if selected_names:
    for p in downloaded_paths:
        base = os.path.basename(p)
        if base in selected_names:
            dest = os.path.join(SELECTED_DIR, base)
            try:
                if os.path.abspath(p) != os.path.abspath(dest):
                    shutil.move(p, dest)
                moved.append(dest)
            except Exception as e:
                print(f"❌ Błąd przenoszenia {base}: {e}")
else:
    # Jeżeli nie było listy – przenieś wszystko
    for p in downloaded_paths:
        base = os.path.basename(p)
        dest = os.path.join(SELECTED_DIR, base)
        try:
            if os.path.abspath(p) != os.path.abspath(dest):
                shutil.move(p, dest)
            moved.append(dest)
        except Exception as e:
            print(f"❌ Błąd przenoszenia {base}: {e}")

print("\n✅ Gotowe. Wybrane pliki dostępne w:")
for p in moved:
    print(f" - {p}")

# (Opcjonalnie) Sprzątanie katalogu tymczasowego z resztek
try:
    for root, _, files in os.walk(DEST_DIR_ALL, topdown=False):
        for f in files:
            fp = os.path.join(root, f)
            if os.path.isfile(fp):
                try:
                    os.remove(fp)
                except Exception:
                    pass
    shutil.rmtree(DEST_DIR_ALL, ignore_errors=True)
except Exception:
    pass

print("\n🎯 Zakończono testy Mega.")

In [ ]:
# Komórka 5: Test Modułu `google_drive.py`

print("="*50)
print("🚀 TEST 3: OPERACJE NA GOOGLE DRIVE 🚀")
print("="*50)
print("UWAGA: Ta komórka wywoła okno autoryzacji Google. Zezwól na dostęp.")

if not gdrive_handler.is_ready():
    print("\n❌ TEST NIEUDANY. Manager Dysku Google nie został poprawnie zainicjalizowany.")
else:
    DUMMY_FILENAME = "test_upload.txt"
    DUMMY_CONTENT = "To jest plik testowy z Colab Universal Downloader."
    file_id_to_delete = None
    
    try:
        # Tworzenie pliku testowego
        with open(DUMMY_FILENAME, "w") as f:
            f.write(DUMMY_CONTENT)
        print(f"Stworzono plik testowy: {DUMMY_FILENAME}")

        # Krok 1: Pobierz ID "Mojego Dysku"
        print("\n--> Testowanie get_drive_id()...")
        parent_id = gdrive_handler.get_drive_id(drive_name="Mój Dysk", is_shared=False)
        assert parent_id == 'root'
        print("✅ ID 'Mojego Dysku' pobrane poprawnie ('root').")

        # Krok 2: Wyślij plik i udostępnij
        print("\n--> Testowanie upload_and_share()...")
        upload_info = gdrive_handler.upload_and_share(DUMMY_FILENAME, parent_id)
        
        if upload_info and upload_info.get('link'):
            print(f"✅ Plik wysłany i udostępniony pomyślnie!")
            print(f"   Link do pobrania: {upload_info['link']}")
            file_id_to_delete = upload_info['id']
        else:
            raise AssertionError("Nie udało się wysłać pliku lub uzyskać linku.")

        # Krok 3: Zaplanuj usunięcie pliku
        if file_id_to_delete:
            print("\n--> Testowanie delete_file_after_delay()...")
            gdrive_handler.delete_file_after_delay(file_id_to_delete, delay_seconds=15)
            print("✅ Zadanie usunięcia pliku zaplanowane na za 15 sekund.")
            print("   Możesz sprawdzić na Dysku Google, czy plik zniknie.")

    except Exception as e:
        print(f"\n❌ WYSTĄPIŁ BŁĄD PODCZAS TESTU: {e}")
    finally:
        # Sprzątanie lokalnego pliku
        if os.path.exists(DUMMY_FILENAME):
            os.remove(DUMMY_FILENAME)
            print(f"\n🧹 Usunięto lokalny plik testowy: {DUMMY_FILENAME}")